In [ ]:
"""
CLI RPG (Learning Module): "Invocation" concepts via a tiny state machine.
"""

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import random


# -----------------------------
# Data model (small + explicit)
# -----------------------------

@dataclass
class Item:
    name: str
    visible: bool = False
    location: str = ""
    found: bool = False
    value: int = 0


@dataclass
class Location:
    name: str
    description: str
    prompt: str
    next_moves: Tuple[str, ...]


@dataclass
class GameState:
    current_location: str
    inventory: List[Item] = field(default_factory=list)
    player_name: str = "PlayerNameHere"


# -----------------------------
# Game data (matches reference)
# -----------------------------

ITEMS: Dict[str, Item] = {
    "boots":  Item(name="boots",  visible=False, location="coastline", found=False, value=0),
    "helmet": Item(name="helmet", visible=False, location="cave",      found=False, value=0),
    "gold":   Item(name="gold",   visible=False, location="shipwreck", found=False, value=10),
    "rope":   Item(name="rope",   visible=False, location="cliffside", found=False, value=0),
}

LOCATIONS: Dict[str, Location] = {
    "coastline": Location(
        name="coastline",
        description=(
            "As you slowly wake up, you find yourself washed up on a shore...\n"
            "You notice a tattered bag in the distance down the coast...\n"
        ),
        prompt="Do you want to head West to search it? (West/No)\n\n",
        next_moves=("shipwreck", "cave", "cliffside"),
    ),
    "shipwreck": Location(
        name="shipwreck",
        description=(
            "You notice your ship wrecked up the coastline.\n"
            "There is no sign of life as far as the eye can see..."
        ),
        prompt="Would you like to travel South to look for any survivors? (South/No)\n",
        next_moves=("coastline", "cave", "cliffside"),
    ),
    "cave": Location(
        name="cave",
        description=(
            "There is an eerie cave a little bit further east.\n"
            "There might be something valuable hidden here..."
        ),
        prompt="Would you like to explore the cave? (East/No)\n",
        next_moves=("coastline", "shipwreck", "cliffside"),
    ),
    "cliffside": Location(
        name="cliffside",
        description=(
            "Ahead is a cliff...\n"
            "This vantage point may be high enough up to search for help...\n"
        ),
        prompt="Would you like to head North to explore? (North/No)\n",
        next_moves=("coastline", "shipwreck", "cave"),
    ),
}

# Directional commands are only valid from the location that prompts them.
DIR_HOME = {
    "west": "coastline",
    "south": "shipwreck",
    "east": "cave",
    "north": "cliffside",
}


# -----------------------------
# Helpers (invocation building blocks)
# -----------------------------

def norm(s: str) -> str:
    return (s or "").strip().lower()

def roll(sides: int) -> int:
    return random.randint(1, sides)

def has_item(state: GameState, item_name: str) -> bool:
    return any(i.name == item_name for i in state.inventory)

def add_item_if_missing(state: GameState, item_name: str) -> None:
    if not has_item(state, item_name):
        base = ITEMS[item_name]
        state.inventory.append(Item(**base.__dict__))

def drop_item(state: GameState, item_name: str) -> bool:
    before = [i.name for i in state.inventory]
    state.inventory = [i for i in state.inventory if i.name != item_name]
    after = [i.name for i in state.inventory]
    print(f"Current inventory before dropping: {before}")
    print(f"New inventory after dropping: {after}")
    return before != after

def print_inventory(state: GameState) -> None:
    if not state.inventory:
        print("You currently have no items.\n")
        return
    print("You currently have the following items:")
    for it in state.inventory:
        print(f"- {it.name}")
    print()

def show_help() -> None:
    print("Available commands:")
    print("- 'inv': Show items in your inventory.")
    print("- 'drop [item]': Remove an item from your inventory.")
    print("- 'help': Show this commands menu.")
    print("- 'proceed': Resume your adventure.")
    print("Type 'proceed' or any other command to proceed with your adventure.\n")


# -----------------------------
# Tutor gate (minimal + diegetic)
# -----------------------------

VAGUE = {
    "idk", "dont know", "don't know", "whatever", "anything", "stuff", "somehow",
    "maybe", "be careful", "careful", "try", "just", "figure it out", "wing it"
}

def tutor_gate(state: GameState, max_attempts: int = 3) -> Optional[Dict[str, str]]:
    def normalize(s: str) -> str:
        return " ".join((s or "").strip().lower().split())

    def has_vague(s: str) -> bool:
        ns = normalize(s)
        return any(v in ns for v in VAGUE)

    def valid_tolerance(t: str) -> bool:
      nt = normalize(t)
      has_level = any(x in nt for x in ("low", "med", "medium", "high"))
      # boundary must be explicit: digit OR "okay losing" OR "not okay" + a token
      has_boundary = (
          any(ch.isdigit() for ch in nt)
          or ("okay" in nt and ("lose" in nt or "losing" in nt))
          or ("not okay" in nt)
      )
      return has_level and has_boundary

    inv = [i.name for i in state.inventory] or ["(empty)"]
    print("\nThe cliffside wind bites. A scratched sign reads: 'STATE YOUR PLAN.'")
    print(f"Inventory on hand: {', '.join(inv)}")
    print("Format (4 lines):")
    print("  O: <objective>")
    print("  R: <resources you will use>")
    print("  T: <risk tolerance: low/med/high + what loss you accept>")
    print("  C: <constraints you must respect>\n")

    for attempt in range(1, max_attempts + 1):
        o = input("O: ").strip()
        r = input("R: ").strip()
        t = input("T: ").strip()
        c = input("C: ").strip()

        plan = {"O": o, "R": r, "T": t, "C": c}

        MINLEN = {"O": 8, "R": 3, "T": 4, "C": 8}  # rope is valid; constraints/objective need more
        for k in ("O", "R", "T", "C"):
            val = plan[k].strip()
            if len(val) < MINLEN[k]:
                if k == "R":
                    print("\nThe sign doesn’t react. Name at least one resource (e.g., rope).\n")
                elif k == "T":
                    print("\nThe sign doesn’t react. T needs low/med/high + a boundary (e.g., 'med; okay losing 5 hp').\n")
                else:
                    print("\nThe sign doesn’t react. Add specifics.\n")
                break
            if has_vague(val):
                print("\nThe sign doesn’t react. That reads like vibes—be concrete.\n")
                break

        else:
            if not valid_tolerance(plan["T"]):
                print("\nThe sign doesn’t react. T must include Low/Med/High AND a clear loss boundary.\n")
                continue

            print("\nThe sign creaks. The climb is yours.\n")
            return plan

        remaining = max_attempts - attempt
        if remaining:
            print(f"Try again ({remaining} left). Keep it tight.\n")

    print("The wind howls. No plan, no climb.\n")
    return None


# -----------------------------
# Location travel prompt
# -----------------------------

def next_location_prompt(state: GameState) -> None:
    loc = LOCATIONS[state.current_location]
    print("Where would you like to go next? Your options are:\n")
    for move in loc.next_moves:
        print(move)
    print("\nChoose a location to travel to next (or type 'stay' to remain here):")
    inp = norm(input("> "))
    if inp == "stay":
        return
    if inp in loc.next_moves:
        state.current_location = inp
        return
    print("Invalid input, please choose a valid location.\n")


# -----------------------------
# Event handlers
# -----------------------------

def handle_west(state: GameState) -> None:
    print("You search the bag...\n")
    r = roll(6)
    print(f"Rolling the dice... You rolled a {r}!\n")
    if r >= 3:
        print("In the bag you find a tattered pair of boots!\n")
        add_item_if_missing(state, "boots")
        print("At least the rest of the journey won't be as rugged...\n")
    else:
        print("You found nothing.\n")
    next_location_prompt(state)

def handle_south(state: GameState) -> None:
    print("As you head towards the ship...\n")
    print("There are no signs of life...\nHowever, in the wreckage, you see a safe that looks to be open.\n")
    print("Do you want to investigate this? (Yes/No)\n")
    inp = norm(input("> "))
    if inp == "yes":
        r = roll(6)
        print(f"Rolling the dice... You rolled a {r}!\n")
        if r >= 3:
            print("You search the safe and find some gold.\n")
            add_item_if_missing(state, "gold")
            print("You place the gold in your pockets...\n")
        else:
            print("You found nothing of interest.\n")
        next_location_prompt(state)
    elif inp == "no":
        print("You decide not to investigate the safe.\n")
        next_location_prompt(state)
    else:
        print("Invalid input, please choose 'yes' or 'no'.\n")
        handle_south(state)

def handle_east(state: GameState) -> None:
    print("As you enter the cave, there is a strange feeling that someone is watching you...\n")
    print("You notice a glint further in the cave...\n")
    print("As you get closer, you make out a helmet still attached to the remains of a previous traveler.\n")
    print("Are you brave enough to search the remains? (Yes/No)\n")
    inp = norm(input("> "))
    if inp == "yes":
        r = roll(6)
        print(f"Rolling the dice... You rolled a {r}!\n")
        if r >= 3:
            print("Searching the remains, you find a helmet!\n")
            add_item_if_missing(state, "helmet")
            print("You think to yourself, this may come in handy later...\n")
        else:
            print("You found nothing of interest.\n")
        next_location_prompt(state)
    elif inp == "no":
        print("You decide not to search the remains.\n")
        next_location_prompt(state)
    else:
        print("Invalid input, please choose 'yes' or 'no'.\n")
        handle_east(state)

def continue_cliffside(state: GameState) -> None:
    print("\nSome time later...\n")
    print("As you reach the cliff, it's time for you to decide if it is worth the risk.\n")
    print("\nWARNING: FAILURE TO PASS THIS DICE ROLL COULD RESULT IN DEATH.\n\n")
    print("Do you wish to continue? (Yes/No)\n")
    inp = norm(input("> "))

    if inp == "yes":
        plan = tutor_gate(state)
        if not plan:
            print("You step back from the edge. Not today.\n")
            next_location_prompt(state)
            return

        r = roll(8)
        if r >= 7:
            print(f"You successfully climb the cliff and see a search party!\n"
                  f"You rolled a {r} to pass the dexterity check!\n\nGAME OVER.\n")
        elif 3 < r <= 6:
            print(f"You fall but a bush breaks your fall... You rolled a {r}!\n\nGAME OVER.\n")
        else:
            print(f"Your low dexterity causes you to slip and fall to your death.\n"
                  f"You rolled a {r}! 4+ is required to survive.\n\nGAME OVER.\n")

        print("Would you like to play again? (Yes/No)\n")
        again = norm(input("> "))
        if again == "yes":
            state.current_location = "coastline"
            state.inventory.clear()
        next_location_prompt(state)

    elif inp == "no":
        print("You decide not to risk climbing the cliff right now.\n")
        next_location_prompt(state)
    else:
        print("Invalid input, try again:\n")
        continue_cliffside(state)

def handle_north(state: GameState) -> None:
    print("During your walk towards the cliff, you notice a rope on the ground.\nWould you like to pick this up? (Yes/No)\n")
    rope_inp = norm(input("> "))
    if rope_inp == "yes":
        print("You pick up the rope with hopes it will make this climb easier.\n")
        add_item_if_missing(state, "rope")
        continue_cliffside(state)
    elif rope_inp == "no":
        print("You decide not to take the rope.\n")
        continue_cliffside(state)
    else:
        print("Invalid input, try again:\n")
        continue_cliffside(state)


# -----------------------------
# Main loop (single loop)
# -----------------------------

def run_game() -> None:
    print("What's your name, adventurer?\n")
    name = input("> ").strip() or "Player"
    state = GameState(current_location="coastline", inventory=[], player_name=name)
    print(f"Welcome, {state.player_name}!\n")
    print("Type 'help' to see the list of commands.\n")

    last_rendered_location: Optional[str] = None

    while True:
        loc = LOCATIONS[state.current_location]
        if last_rendered_location != state.current_location:
            print(loc.description)
            print()
            print(loc.prompt)
            last_rendered_location = state.current_location

        raw = input("Type help for a list of commands, enter a command, or follow the directional prompts above: ")
        parts = norm(raw).split()
        command = parts[0] if parts else ""
        args = parts[1:] if len(parts) > 1 else []

        # Directional commands only invoke from the correct location
        if command in ("west", "north", "south", "east"):
            required = DIR_HOME[command]
            if state.current_location != required:
                print(f"You can't go {command} from here.\n")
                next_location_prompt(state)
                continue

            if command == "west":
                handle_west(state)
            elif command == "south":
                handle_south(state)
            elif command == "east":
                handle_east(state)
            elif command == "north":
                handle_north(state)
            continue

        elif command == "inv":
            print_inventory(state)

        elif command == "drop":
            if not args:
                print("You need to specify an item to drop.\n")
            else:
                item_name = " ".join(args)
                removed = drop_item(state, item_name)
                if not removed:
                    print(f"You do not have '{item_name}'.\n")

        elif command == "no":
            next_location_prompt(state)

        elif command == "help":
            show_help()

        elif command == "proceed":
            pass

        else:
            print("Invalid command, try again.\n")


run_game()